In [1]:
!pip install transformers  datasets accelerate -q

In [2]:
import pandas as pd
df = pd.read_csv('train.csv')

In [3]:
from datasets import Dataset

label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
df['labels'] = df[label_cols].values.tolist()

In [4]:
dataset = Dataset.from_pandas(df[['comment_text','labels']])


In [5]:
from sklearn.model_selection import train_test_split
dataset = dataset.train_test_split(test_size=0.2,seed=42)

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['comment_text', 'labels'],
        num_rows: 127656
    })
    test: Dataset({
        features: ['comment_text', 'labels'],
        num_rows: 31915
    })
})

In [7]:
from transformers import AutoTokenizer

In [8]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
sample_text = "This is unbeleiveably frustrating and stupid"
tokens  = tokenizer.tokenize(sample_text)
print(tokens)

['this', 'is', 'un', '##bel', '##ei', '##ve', '##ably', 'frustrating', 'and', 'stupid']


In [10]:
def tokenize_function(examples):
  return tokenizer(
      examples['comment_text'],
      padding='max_length',
      truncation=True,
      max_length=256
  )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
print(tokenized_dataset)

Map:   0%|          | 0/127656 [00:00<?, ? examples/s]

Map:   0%|          | 0/31915 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['comment_text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 127656
    })
    test: Dataset({
        features: ['comment_text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 31915
    })
})


In [ ]:
lengths = df['comment_text'].apply(lambda x: len(tokenizer.tokenize(str(x))))
print("Mean token length:", lengths.mean())
print("Median token length:", lengths.median())
print("% of comments over 128 tokens:", (lengths > 128).mean() * 100)
print("% of comments over 256 tokens:", (lengths > 256).mean() * 100)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (629 > 512). Running this sequence through the model will result in indexing errors


Mean token length: 92.89186004975842
Median token length: 50.0
% of comments over 128 tokens: 18.911957686547055
% of comments over 256 tokens: 6.848362171071185


In [12]:
from transformers import AutoModelForSequenceClassification
import torch
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased',num_labels=6,problem_type='multi_label_classification')

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print(tokenized_dataset['train'].features['labels'])

List(Value('int64'))


In [13]:
from datasets import Sequence, Value

tokenized_dataset = tokenized_dataset.cast_column("labels", Sequence(Value("float32")))

sample = tokenized_dataset['train'][0]
print(type(sample['labels']), sample['labels'])

Casting the dataset:   0%|          | 0/127656 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/31915 [00:00<?, ? examples/s]

<class 'list'> [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [ ]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
)

trainer.train()


Epoch,Training Loss,Validation Loss


In [14]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained('/content/drive/MyDrive/content-moderation-model')
tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/content-moderation-model')

print("Model loaded successfully")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully


In [17]:
import torch

sample_texts = list(df['comment_text'][:10])  # first 10 comments as a test

inputs = tokenizer(sample_texts, padding=True, truncation=True, max_length=256, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
probs = torch.sigmoid(logits)

label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

for i, text in enumerate(sample_texts):
    print(f"\nText: {text[:80]}...")
    for label, score in zip(label_cols, probs[i]):
        if score > 0.5:
            print(f"  {label}: {score:.3f}")


Text: Explanation
Why the edits made under my username Hardcore Metallica Fan were rev...

Text: D'aww! He matches this background colour I'm seemingly stuck with. Thanks.  (tal...

Text: Hey man, I'm really not trying to edit war. It's just that this guy is constantl...

Text: "
More
I can't make any real suggestions on improvement - I wondered if the sect...

Text: You, sir, are my hero. Any chance you remember what page that's on?...

Text: "

Congratulations from me as well, use the tools well.  · talk "...

Text: COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK...
  toxic: 0.994
  obscene: 0.976
  insult: 0.780

Text: Your vandalism to the Matt Shirvington article has been reverted.  Please don't ...

Text: Sorry if the word 'nonsense' was offensive to you. Anyway, I'm not intending to ...

Text: alignment on this subject and which are contrary to those of DuLithgow...
